In [ ]:
!pip install -q "dbt-core==1.10.23" "dbt-duckdb==1.9.6" "kagglehub[pandas-datasets]" duckdb

In [ ]:
from google.colab import files
import os, shutil

uploaded = files.upload()                 # choose kaggle.json
fname = next(iter(uploaded))              # handles names like "kaggle (1).json"

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
shutil.move(fname, os.path.expanduser("~/.kaggle/kaggle.json"))
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
print("Kaggle credentials set")

Saving kaggle.json to kaggle.json
Kaggle credentials set


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

NOTEBOOK_DIR = "/content/drive/MyDrive/Koolid/Tartu Ülikool - Andmeteadus/Data engineering"
PROJECT_DIR  = f"{NOTEBOOK_DIR}/spotify_dbt_duckdb"     # dbt project (saved in Drive)
DATA_DIR     = "/content/data"                          # bronze layer: Parquet files
DUCKDB_PATH  = "/content/spotify.duckdb"                # DuckDB database with the dbt views

# dbt reads these in profiles.yml and _sources.yml
os.environ["SPOTIFY_DATA_DIR"]    = DATA_DIR
os.environ["DUCKDB_PATH"]         = DUCKDB_PATH
os.environ["DUCKDB_MEMORY_LIMIT"] = "6GB"               # Colab has ~12 GB RAM

os.makedirs(DATA_DIR, exist_ok=True)
for d in ["models/staging", "models/intermediate", "models/marts", "tests"]:
    os.makedirs(f"{PROJECT_DIR}/{d}", exist_ok=True)
os.chdir(PROJECT_DIR)                                    # %%writefile and dbt below run from here
print(os.getcwd())

Mounted at /content/drive
/content/drive/MyDrive/Koolid/Tartu Ülikool - Andmeteadus/Data engineering/spotify_dbt_duckdb


# Bronze layer: download from Kaggle and convert to Parquet

In [ ]:
import kagglehub, duckdb

HANDLE = "gonzalopezgil/spotify-charts-daily-updated"   # add "/versions/<n>" to pin a version
FILES  = ["songs", "artists", "charts_songs_daily", "charts_artists_daily"]

con = duckdb.connect()
con.sql("SET memory_limit = '6GB'")
con.sql("SET preserve_insertion_order = false")
con.sql("SET temp_directory = '/content/duckdb_tmp'")

for name in FILES:
    parquet = f"{DATA_DIR}/{name}.parquet"
    if os.path.exists(parquet):
        print(f"{name}: already converted")
        continue
    csv = kagglehub.dataset_download(HANDLE, path=f"{name}.csv")
    con.sql(f"COPY (SELECT * FROM read_csv_auto('{csv}')) TO '{parquet}' (FORMAT parquet)")
    rows = con.sql(f"SELECT COUNT(*) FROM '{parquet}'").fetchone()[0]
    print(f"{name}: {rows:,} rows")
con.close()

100%|██████████| 41.8M/41.8M [00:00<00:00, 107MB/s]


songs: 216,601 rows


100%|██████████| 4.22M/4.22M [00:00<00:00, 154MB/s]

artists: 74,162 rows


100%|██████████| 10.3G/10.3G [01:36<00:00, 114MB/s]


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

charts_songs_daily: 44,969,351 rows


100%|██████████| 2.84G/2.84G [00:27<00:00, 109MB/s] 


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

charts_artists_daily: 24,995,340 rows


In [ ]:
import duckdb

DEMO_DB = "/content/spotify_demo.duckdb"     # separate from dbt's database, so they don't collide
con = duckdb.connect(DEMO_DB)
con.sql("SET memory_limit = '6GB'")
con.sql("SET threads = 2")
con.sql("SET preserve_insertion_order = false")
con.sql("SET temp_directory = '/content/duckdb_tmp'")

# Bronze: one view per Parquet file, named like the dbt source tables
for name in ["songs", "artists", "charts_songs_daily", "charts_artists_daily"]:
    con.sql(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_parquet('{DATA_DIR}/{name}.parquet')")

## Stg layer

### stg_songs

In [ ]:
# MODEL NAME: stg_songs.sql
#Staging: one row per song. Only renames and data types.

con.sql("""
CREATE OR REPLACE VIEW stg_songs AS
SELECT
    track_uri -- the song's main URI, used as song_id downstream
    , track_name
    , artist_names  -- '|'-separated
    , artist_uris -- '|'-separated
    , NULLIF(TRIM(label), '')  AS label -- normalise names and also assign explicit NULL if name missing
    , TRY_CAST(release_date AS DATE)  AS release_date
    , all_uris   -- '|'-separated list of every URI of this song
FROM songs
""")

### stg_artists

In [ ]:
# MODEL NAME: stg_artists.sql
# Staging: one row per artist. Only renames and data types.
# Note: names like 'N/A' and 'None' are real values in the source, not NULLs.


con.sql("""
CREATE OR REPLACE VIEW stg_artists AS
SELECT
    artist_uri
    , artist_name
    , CAST(monthly_listeners AS BIGINT)                 AS monthly_listeners
    , CAST(monthly_listeners_rank AS INTEGER)           AS monthly_listeners_rank
    , CAST(monthly_listeners_peak_rank AS INTEGER)      AS monthly_listeners_peak_rank
    , CAST(monthly_listeners_peak_listeners AS BIGINT)  AS monthly_listeners_peak_listeners
FROM artists
""")

### stg_charts_songs_daily

In [ ]:
# MODEL NAME: stg_charts_songs_daily.sql
# Staging: one row per source row. Only renames and data types, no business logic.
# The only filter is the analysis window (var start_date), so downstream models read 2025+ only.

con.sql("""
CREATE OR REPLACE VIEW stg_charts_songs_daily AS
SELECT
    CAST(date AS DATE)                              AS chart_date
    , LOWER(country)                                AS country_code      -- 'global' = worldwide chart
    , CAST(rank AS INTEGER)                         AS chart_rank
    , uri                                           AS track_uri
    , track_name
    , artist_names                                  -- '|'-separated
    , artist_uris                                   -- '|'-separated
    , NULLIF(TRIM(label), '')                       AS label -- same as for song names in dim
    , CAST(streams AS BIGINT)                       AS streams -- for this day
    , CAST(peak_rank AS INTEGER)                    AS peak_rank
    , NULLIF(CAST(previous_rank AS INTEGER), -1)    AS previous_rank     -- source uses -1 for "not on chart yesterday"
    , CAST(days_on_chart AS INTEGER)                AS days_on_chart
    , CAST(consecutive_days AS INTEGER)             AS consecutive_days
    , entry_status
    , CAST(peak_date AS DATE)                       AS peak_date
    , CAST(entry_rank AS INTEGER)                   AS entry_rank
    , CAST(entry_date AS DATE)                      AS entry_date
    , TRY_CAST(release_date AS DATE)                AS release_date      -- date of the album the track is linked to now
FROM charts_songs_daily
WHERE 0=0
  AND CAST(date AS DATE) >= DATE '2026-01-01'
    --AND CAST(date AS DATE) >= DATE '{{ var("start_date") }}'
""")

In [ ]:
# Näide, kuidas näha nende mudelite sisse:

con.sql("""
SELECT *
FROM stg_charts_songs_daily
WHERE 0=0
  AND artist_names = 'Billie Eilish'
  AND country_code = 'us'
  AND track_name = 'WILDFLOWER'
  AND chart_date BETWEEN DATE '2026-01-28' AND DATE '2026-02-03'
ORDER BY chart_date
""").df()

,chart_date,country_code,chart_rank,track_uri,track_name,artist_names,artist_uris,label,streams,peak_rank,previous_rank,days_on_chart,consecutive_days,entry_status,peak_date,entry_rank,entry_date,release_date
0,2026-01-28,us,110,spotify:track:3QaPy1KgI7nu9FJEQUgn6h,WILDFLOWER,Billie Eilish,spotify:artist:6qqNVTkY8uBg9cP3Jd7DAH,Darkroom/Interscope Records,380524,10,114,620,34,MOVED_UP,2024-05-17,10,2024-05-17,2024-05-17
1,2026-01-29,us,110,spotify:track:3QaPy1KgI7nu9FJEQUgn6h,WILDFLOWER,Billie Eilish,spotify:artist:6qqNVTkY8uBg9cP3Jd7DAH,Darkroom/Interscope Records,384581,10,110,621,35,NO_CHANGE,2024-05-17,10,2024-05-17,2024-05-17
2,2026-01-30,us,141,spotify:track:3QaPy1KgI7nu9FJEQUgn6h,WILDFLOWER,Billie Eilish,spotify:artist:6qqNVTkY8uBg9cP3Jd7DAH,Darkroom/Interscope Records,375413,10,110,622,36,MOVED_DOWN,2024-05-17,10,2024-05-17,2024-05-17
3,2026-01-31,us,144,spotify:track:3QaPy1KgI7nu9FJEQUgn6h,WILDFLOWER,Billie Eilish,spotify:artist:6qqNVTkY8uBg9cP3Jd7DAH,Darkroom/Interscope Records,347757,10,141,623,37,MOVED_DOWN,2024-05-17,10,2024-05-17,2024-05-17
4,2026-02-01,us,126,spotify:track:3QaPy1KgI7nu9FJEQUgn6h,WILDFLOWER,Billie Eilish,spotify:artist:6qqNVTkY8uBg9cP3Jd7DAH,Darkroom/Interscope Records,347508,10,144,624,38,MOVED_UP,2024-05-17,10,2024-05-17,2024-05-17
5,2026-02-02,us,30,spotify:track:3QaPy1KgI7nu9FJEQUgn6h,WILDFLOWER,Billie Eilish,spotify:artist:6qqNVTkY8uBg9cP3Jd7DAH,Darkroom/Interscope Records,702731,10,126,625,39,MOVED_UP,2024-05-17,10,2024-05-17,2024-05-17
6,2026-02-03,us,26,spotify:track:3QaPy1KgI7nu9FJEQUgn6h,WILDFLOWER,Billie Eilish,spotify:artist:6qqNVTkY8uBg9cP3Jd7DAH,Darkroom/Interscope Records,731603,10,30,626,40,MOVED_UP,2024-05-17,10,2024-05-17,2024-05-17


### stg_charts_artists_daily

In [ ]:
# MODEL NAME: stg_charts_artists_daily.sql
# Staging: one row per source row. Only renames and data types, no business logic.
# The only filter is the analysis window (var start_date), so downstream models read 2025+ only.

con.sql("""
CREATE OR REPLACE VIEW stg_charts_artists_daily AS
SELECT
    CAST(date AS DATE)                              AS chart_date
    , LOWER(country)                                AS country_code      -- 'global' = worldwide chart
    , CAST(rank AS INTEGER)                         AS chart_rank
    , uri                                           AS artist_uri
    , artist_name
    , CAST(peak_rank AS INTEGER)                    AS peak_rank
    , NULLIF(CAST(previous_rank AS INTEGER), -1)    AS previous_rank     -- source uses -1 for "not on chart yesterday"
    , CAST(days_on_chart AS INTEGER)                AS days_on_chart
    , CAST(consecutive_days AS INTEGER)             AS consecutive_days
    , entry_status
    , CAST(peak_date AS DATE)                       AS peak_date
    , CAST(entry_rank AS INTEGER)                   AS entry_rank
    , CAST(entry_date AS DATE)                      AS entry_date
FROM charts_artists_daily
WHERE 0=0
  AND CAST(date AS DATE) >= DATE '2026-01-01'
    --AND CAST(date AS DATE) >= DATE '{{ var("start_date") }}'
""")

# Intermediate (silver) layer

### int_song_uri_map

In [ ]:
con.sql("""
-- Create a mapping table of the main uri and all other uris of a song. As many rows per song as there are uris on all_uris
    SELECT
        UNNEST(STRING_SPLIT(COALESCE(s.all_uris, s.track_uri), '|'))   AS track_uri
        , s.track_uri                                                   AS song_id -- the main uri
    FROM stg_songs s
    WHERE s.track_uri = 'spotify:track:36eEND5G3RbI4XD7Xupnmv'
""").df()

,track_uri,song_id
0,spotify:track:36eEND5G3RbI4XD7Xupnmv,spotify:track:36eEND5G3RbI4XD7Xupnmv
1,spotify:track:0L4SSGbkayyR1RxePf4T4C,spotify:track:36eEND5G3RbI4XD7Xupnmv


In [ ]:
# MODEL NAME: int_song_uri_map.sql
# One row per Spotify track URI -> the song it belongs to (song_id).
# Spotify gives one song several URIs (e.g. single vs album release). songs.all_uris lists them all,
# and the chart switches between them (discovery: "Distance" charted under 2 URIs).
# Discovery: every URI belongs to exactly one song, so the QUALIFY below is only a safety net.

## NEEDS TO BE A VIEW OR A TABLE WITH FULL SCAN in case track_uri changes

con.sql("""
CREATE OR REPLACE VIEW int_song_uri_map AS
-- Map every URI of a song to its main URI. One row per URI in all_uris.
SELECT
    UNNEST(STRING_SPLIT(COALESCE(s.all_uris, s.track_uri), '|'))   AS track_uri
    , s.track_uri                                                   AS song_id      -- the main URI
FROM stg_songs s
""")

### int_charts_songs_daily

In [ ]:
# MODEL NAME: int_charts_songs_daily.sql
# Daily song chart rows (since 2026 FOR NOW), cleaned. One row per chart_date x country x track_uri.
# Two fixes from discovery:
#   1) Duplicate records: the same URI twice on one chart day at different ranks, always with
#      identical streams (192 cases). They are one entry recorded twice -> keep the better rank.
#   2) song_id: different URIs of the same song get one song_id, so they can be combined later.
#      Those rows are NOT duplicates (different streams, own chart history) and are kept --> THESE ARE COMBINED LATER in int_songs_weekly
          # Examples here Re-release or remaster, Single and album version etc

con.sql("""
CREATE OR REPLACE VIEW int_charts_songs_daily AS
WITH deduplicated AS (
    SELECT
        c.chart_date
        , c.country_code
        , c.chart_rank
        , c.track_uri
        , c.track_name
        , c.artist_names
        , c.artist_uris
        , c.label
        , c.streams
        , c.peak_rank
        , c.previous_rank
        , c.days_on_chart
        , c.consecutive_days
        , c.entry_status
        , c.peak_date
        , c.entry_rank
        , c.entry_date
        , c.release_date
    FROM stg_charts_songs_daily c
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY c.chart_date, c.country_code, c.track_uri
        ORDER BY c.chart_rank) = 1
)

-- Chart rows only have the URI used on that day. Join int_song_uri_map to add song_id,
-- so all URIs of a song share one ID and their histories can be combined downstream.
SELECT
    d.chart_date
    , CAST(DATE_TRUNC('week', d.chart_date) AS DATE)    AS week_start        -- Monday of the ISO week
    , d.country_code
    , d.chart_rank
    , d.track_uri
    , COALESCE(m.song_id, d.track_uri)            AS song_id -- URIs missing from songs.csv keep their own URI, can happen if song later removed from Spotify
    , d.track_name
    , d.artist_names
    , d.label
    , d.streams
    , d.entry_date
    , d.release_date
FROM deduplicated d
    LEFT JOIN int_song_uri_map m ON d.track_uri = m.track_uri
""")

### Dealing with duplicate artist chart entries

In [ ]:
con.sql("""
SELECT
    c.chart_date
    , c.country_code
    , c.artist_uri
    , c.artist_name
    , c.chart_rank
    , c.days_on_chart
    , c.entry_date
    , c.entry_status
FROM stg_charts_artists_daily c
QUALIFY COUNT(*) OVER (PARTITION BY c.chart_date, c.country_code, c.artist_uri) > 1 -- artistid, kellel ühel reporting päeval mitu sama URIga entry-t
ORDER BY
    c.artist_uri
    , c.chart_date
    , c.chart_rank
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,chart_date,country_code,artist_uri,artist_name,chart_rank,days_on_chart,entry_date,entry_status
0,2026-08-10,ro,spotify:artist:0lD0cnzSrUjThgH9YxBF82,Luis Gabriel,53,1523,2021-11-05,MOVED_DOWN
1,2026-08-10,ro,spotify:artist:0lD0cnzSrUjThgH9YxBF82,Luis Gabriel,62,318,2025-09-26,NO_CHANGE
2,2026-08-11,ro,spotify:artist:0lD0cnzSrUjThgH9YxBF82,Luis Gabriel,58,1524,2021-11-05,MOVED_DOWN
3,2026-08-11,ro,spotify:artist:0lD0cnzSrUjThgH9YxBF82,Luis Gabriel,60,319,2025-09-26,MOVED_UP
4,2026-08-12,ro,spotify:artist:0lD0cnzSrUjThgH9YxBF82,Luis Gabriel,60,1525,2021-11-05,MOVED_DOWN
...,...,...,...,...,...,...,...,...
63,2026-08-17,za,spotify:artist:6wq5brCC7NSFUDJ8rL0fcF,Inkos'yamagcokama,25,8,2026-08-10,MOVED_DOWN
64,2026-08-18,za,spotify:artist:6wq5brCC7NSFUDJ8rL0fcF,Inkos'yamagcokama,2,687,2022-12-03,NO_CHANGE
65,2026-08-18,za,spotify:artist:6wq5brCC7NSFUDJ8rL0fcF,Inkos'yamagcokama,43,9,2026-08-10,MOVED_DOWN
66,2026-01-21,vn,spotify:artist:7yquVKfxBuNFJbG9cy2R8A,VCT,48,1554,2021-10-21,MOVED_DOWN


In [ ]:
# NÄIDE 1 (Luis Gabriel) sama URIga on 2 artisti chartis augustis ja septembris
con.sql("""
SELECT
    c.chart_date
    , c.country_code
    , c.artist_uri
    , c.artist_name
    , c.chart_rank
    , c.days_on_chart
    , c.entry_date
    , c.entry_status
    , COUNT(*) OVER (PARTITION BY c.chart_date, c.country_code, c.artist_uri) AS rows_per_reporting_date
FROM stg_charts_artists_daily c
WHERE 0=0
  AND c.artist_uri = 'spotify:artist:0lD0cnzSrUjThgH9YxBF82'
  AND c.country_code = 'ro'
QUALIFY rows_per_reporting_date > 1
ORDER BY
    c.artist_uri
    , c.chart_date
    , c.chart_rank
""").df()

,chart_date,country_code,artist_uri,artist_name,chart_rank,days_on_chart,entry_date,entry_status,rows_per_reporting_date
0,2026-08-10,ro,spotify:artist:0lD0cnzSrUjThgH9YxBF82,Luis Gabriel,53,1523,2021-11-05,MOVED_DOWN,2
1,2026-08-10,ro,spotify:artist:0lD0cnzSrUjThgH9YxBF82,Luis Gabriel,62,318,2025-09-26,NO_CHANGE,2
2,2026-08-11,ro,spotify:artist:0lD0cnzSrUjThgH9YxBF82,Luis Gabriel,58,1524,2021-11-05,MOVED_DOWN,2
3,2026-08-11,ro,spotify:artist:0lD0cnzSrUjThgH9YxBF82,Luis Gabriel,60,319,2025-09-26,MOVED_UP,2
4,2026-08-12,ro,spotify:artist:0lD0cnzSrUjThgH9YxBF82,Luis Gabriel,60,1525,2021-11-05,MOVED_DOWN,2
5,2026-08-12,ro,spotify:artist:0lD0cnzSrUjThgH9YxBF82,Luis Gabriel,63,320,2025-09-26,MOVED_DOWN,2
6,2026-08-13,ro,spotify:artist:0lD0cnzSrUjThgH9YxBF82,Luis Gabriel,61,1526,2021-11-05,MOVED_DOWN,2
7,2026-08-13,ro,spotify:artist:0lD0cnzSrUjThgH9YxBF82,Luis Gabriel,64,321,2025-09-26,MOVED_DOWN,2
8,2026-08-14,ro,spotify:artist:0lD0cnzSrUjThgH9YxBF82,Luis Gabriel,60,1527,2021-11-05,MOVED_UP,2
9,2026-08-14,ro,spotify:artist:0lD0cnzSrUjThgH9YxBF82,Luis Gabriel,63,322,2025-09-26,MOVED_UP,2


In [ ]:
# NÄIDE 2 (Inkos'yamagcokama) sama URIga on 2 artisti chartis augustis ja septembris
con.sql("""
SELECT
    c.chart_date
    , c.country_code
    , c.artist_uri
    , c.artist_name
    , c.chart_rank
    , c.days_on_chart
    , c.entry_date
    , c.entry_status
    , COUNT(*) OVER (PARTITION BY c.chart_date, c.country_code, c.artist_uri) AS rows_per_reporting_date
FROM stg_charts_artists_daily c
WHERE 0=0
  AND c.artist_uri = 'spotify:artist:6wq5brCC7NSFUDJ8rL0fcF'
  --AND c.country_code = 'ro'
QUALIFY rows_per_reporting_date > 1
ORDER BY
    c.artist_uri
    , c.chart_date
    , c.chart_rank
""").df()

,chart_date,country_code,artist_uri,artist_name,chart_rank,days_on_chart,entry_date,entry_status,rows_per_reporting_date
0,2026-08-10,za,spotify:artist:6wq5brCC7NSFUDJ8rL0fcF,Inkos'yamagcokama,2,679,2022-12-03,MOVED_DOWN,2
1,2026-08-10,za,spotify:artist:6wq5brCC7NSFUDJ8rL0fcF,Inkos'yamagcokama,15,1,2026-08-10,NEW_ENTRY,2
2,2026-08-11,za,spotify:artist:6wq5brCC7NSFUDJ8rL0fcF,Inkos'yamagcokama,2,680,2022-12-03,NO_CHANGE,2
3,2026-08-11,za,spotify:artist:6wq5brCC7NSFUDJ8rL0fcF,Inkos'yamagcokama,18,2,2026-08-10,MOVED_DOWN,2
4,2026-08-12,za,spotify:artist:6wq5brCC7NSFUDJ8rL0fcF,Inkos'yamagcokama,2,681,2022-12-03,NO_CHANGE,2
5,2026-08-12,za,spotify:artist:6wq5brCC7NSFUDJ8rL0fcF,Inkos'yamagcokama,25,3,2026-08-10,MOVED_DOWN,2
6,2026-08-13,za,spotify:artist:6wq5brCC7NSFUDJ8rL0fcF,Inkos'yamagcokama,2,682,2022-12-03,NO_CHANGE,2
7,2026-08-13,za,spotify:artist:6wq5brCC7NSFUDJ8rL0fcF,Inkos'yamagcokama,26,4,2026-08-10,MOVED_DOWN,2
8,2026-08-14,za,spotify:artist:6wq5brCC7NSFUDJ8rL0fcF,Inkos'yamagcokama,2,683,2022-12-03,NO_CHANGE,2
9,2026-08-14,za,spotify:artist:6wq5brCC7NSFUDJ8rL0fcF,Inkos'yamagcokama,18,5,2026-08-10,MOVED_UP,2


In [ ]:
# NÄIDE 3 (VCT) sama URIga on 2 artisti chartis Jan- Feb
# This artist has 1 day (21.01.26 with an additional "side history") with double same URI entries
  # But the "side history" alone is in the chart 14 days
con.sql("""
SELECT
    c.chart_date
    , c.country_code
    , c.artist_uri
    , c.artist_name
    , c.chart_rank
    , c.days_on_chart
    , c.entry_date
    , c.entry_status
    , COUNT(*) OVER (PARTITION BY c.chart_date, c.country_code, c.artist_uri) AS rows_per_reporting_date
FROM stg_charts_artists_daily c
WHERE 0=0
  AND c.artist_uri = 'spotify:artist:7yquVKfxBuNFJbG9cy2R8A'
  --AND c.country_code = 'ro'
QUALIFY rows_per_reporting_date > 1
ORDER BY
    c.artist_uri
    , c.chart_date
    , c.chart_rank
""").df()

,chart_date,country_code,artist_uri,artist_name,chart_rank,days_on_chart,entry_date,entry_status,rows_per_reporting_date
0,2026-01-21,vn,spotify:artist:7yquVKfxBuNFJbG9cy2R8A,VCT,48,1554,2021-10-21,MOVED_DOWN,2
1,2026-01-21,vn,spotify:artist:7yquVKfxBuNFJbG9cy2R8A,VCT,53,1,2026-01-21,NEW_ENTRY,2


In [ ]:
# NÄIDE 1 (Luis Gabriel) Here we see that the same artist name is linked to different URIs for different periods
# the second artist’s history moved to the wrong URI on 2026-08-10.

con.sql("""
SELECT
    c.artist_uri
    , c.entry_date
    , MIN(c.chart_date)     AS first_seen
    , MAX(c.chart_date)     AS last_seen
    , COUNT(*)              AS chart_days
    , MIN(c.chart_rank)     AS best_rank
FROM stg_charts_artists_daily c
WHERE 0=0
    AND c.artist_name = 'Luis Gabriel'
    AND c.country_code = 'ro'
GROUP BY ALL
ORDER BY first_seen
""").df()

,artist_uri,entry_date,first_seen,last_seen,chart_days,best_rank
0,spotify:artist:03bnhSwfwaSQhZyVy9m6ob,2025-09-26,2026-01-01,2026-08-09,209,33
1,spotify:artist:0lD0cnzSrUjThgH9YxBF82,2021-11-05,2026-01-01,2026-10-25,286,23
2,spotify:artist:0lD0cnzSrUjThgH9YxBF82,2025-09-26,2026-08-10,2026-09-02,24,58


In [ ]:
# NÄIDE 2 (Inkos'yamagcokama) Here we see that the same artist name is linked to different URIs for different periods
# the side history appeared on 2026-08-10.
con.sql("""
SELECT
    c.artist_uri
    , c.entry_date
    , MIN(c.chart_date)     AS first_seen
    , MAX(c.chart_date)     AS last_seen
    , COUNT(*)              AS chart_days
    , MIN(c.chart_rank)     AS best_rank
FROM stg_charts_artists_daily c
WHERE 0=0
    AND c.artist_name LIKE '%Inkos%'
    AND c.country_code = 'za'
GROUP BY ALL
ORDER BY first_seen
""").df()

,artist_uri,entry_date,first_seen,last_seen,chart_days,best_rank
0,spotify:artist:6wq5brCC7NSFUDJ8rL0fcF,2022-12-03,2026-01-01,2026-10-25,161,1
1,spotify:artist:6wq5brCC7NSFUDJ8rL0fcF,2026-08-10,2026-08-10,2026-08-18,9,12


In [ ]:
# NÄIDE 3 (VCT) Here we see that the same artist name is linked to different URIs for different periods
#
con.sql("""
SELECT
    c.artist_uri
    , c.entry_date
    , MIN(c.chart_date)     AS first_seen
    , MAX(c.chart_date)     AS last_seen
    , COUNT(*)              AS chart_days
    , MIN(c.chart_rank)     AS best_rank
FROM stg_charts_artists_daily c
WHERE 0=0
    AND c.artist_name = 'VCT'
    AND c.country_code = 'vn'
GROUP BY ALL
ORDER BY first_seen
""").df()

,artist_uri,entry_date,first_seen,last_seen,chart_days,best_rank
0,spotify:artist:7yquVKfxBuNFJbG9cy2R8A,2021-10-21,2026-01-01,2026-08-09,195,14
1,spotify:artist:2FO4DJ1tBx3Y7sYJqOZMsw,2026-01-21,2026-01-21,2026-03-15,43,89
2,spotify:artist:7yquVKfxBuNFJbG9cy2R8A,2026-01-21,2026-01-21,2026-02-04,15,41


In [ ]:
con.sql("""
WITH double_rows AS (
    SELECT
        c.*
    FROM stg_charts_artists_daily c
    QUALIFY COUNT(*) OVER (PARTITION BY c.chart_date, c.country_code, c.artist_uri) > 1
)

SELECT
    d.artist_uri
    , ANY_VALUE(d.artist_name)          AS artist_name
    , d.country_code
    , MIN(d.chart_date)                 AS first_double_day
    , MAX(d.chart_date)                 AS last_double_day
    , COUNT(DISTINCT d.chart_date)      AS double_days
FROM double_rows d
GROUP BY ALL
ORDER BY first_double_day
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,artist_uri,artist_name,country_code,first_double_day,last_double_day,double_days
0,spotify:artist:7yquVKfxBuNFJbG9cy2R8A,VCT,vn,2026-01-21,2026-01-21,1
1,spotify:artist:6wq5brCC7NSFUDJ8rL0fcF,Inkos'yamagcokama,za,2026-08-10,2026-08-18,9
2,spotify:artist:0lD0cnzSrUjThgH9YxBF82,Luis Gabriel,ro,2026-08-10,2026-09-02,24


**Problem**

In the artist chart, the source sometimes lists a second artist with the same name under another artist’s URI. On some days the same URI then appears twice in one country, with two different chart histories: different entry_date values, days_on_chart counters and ranks. The longer history is the real artist. The shorter one, the side history, belongs to someone else. In 2026 there are 3 such cases:

- Luis Gabriel: the second artist’s history moved to the wrong URI on 2026-08-10. From 2026-08-10 to 2026-09-02, the second Luis Gabriel’s chart entry (entry_date 2025-09-26, previously under their own URI 03bnh…) appears under the first Luis Gabriel’s URI (0lD0…).
  - 2 different artists, Both URIs chart in parallel for months, each with its own history, until the second one’s rows move to the wrong URI.


- Inkos’yamagcokama: the side history appeared on 2026-08-10. On 2026-08-10, a second chart entry starts as a NEW_ENTRY under the same URI and lasts 9 days. It never appears under another URI, so the right artist can’t be identified.
  - Probably 2 different artists, but the second one’s real URI never appears, so it can’t be proven from the data.


- VCT (vn): on 2026-01-21 two new “VCT” chart entries start. One appears under its own URI (2FO4…, ranks 134–161). The other appears under the main VCT’s URI (7yqu…, ranks 41–53) as a side history. On 2026-01-21 the side history and the main history are on the chart together; on the next 14 days the side history is alone under 7yqu…. The two new entries have the same entry_date but different ranks, so they’re different chart entries, not one entry listed twice.
  - At least 2, possibly 3. The two new entries are at different ranks on the same days, so they’re separate entries. Who the side entry really is can’t be identified.


**PROPOSAL**: It would manually checking each duplicate to actually undersand if the entries are different artists with the same name, where Spotify has messed up URI mapping or double URI entries for the same artist. Therefore, our suggested approach is: The model only looks at one URI and country at a time. If the URI appears twice on any day, it keeps the history with the most chart days and deletes every row of the other histories.

Impact on the 3 cases:
- Luis Gabriel: keep main Luis (URI= 0lD0…, entry_date = 2021-11-05), delete extra Luis (URI= 0lD0…, entry_date = 2025-09-26), keep the new Luis (URI= 03bnh…, entry_date = 2021-09-26) it never overlaps with main Luis.
- Inkos’yamagcokama: Keep main (entry in 2022), delete additional (same URI, later entry)
- VCT: keep main VCT (URI= 7yqu…, entry_date = 2021-10-21), delete extra VCT (URI= 7yqu…, entry_date = 2026-01-21), keep the new VCT (URI= 2FO4… entry_date = 2026-01-21) it never overlaps with main VCT.


**Main profile is decided based on the earliest date in the chart (if equal, then days in the chart is used for tie-break).**

In [ ]:
con.sql("""
SELECT * FROM stg_charts_artists_daily LIMIT 10
""").df()

,chart_date,country_code,chart_rank,artist_uri,artist_name,peak_rank,previous_rank,days_on_chart,consecutive_days,entry_status,peak_date,entry_rank,entry_date
0,2026-05-03,ae,87,spotify:artist:5abSRg0xN1NV3gLbuvX24M,Amr Diab,15,87,951,615,NO_CHANGE,2025-07-06,155,2021-10-21
1,2026-05-03,ae,88,spotify:artist:7CajNmpbOovFoOoasH2HaY,Calvin Harris,15,86,1654,16,MOVED_DOWN,2024-10-26,94,2021-10-21
2,2026-05-03,ae,89,spotify:artist:2oSONSC9zQ4UonDKnLqksx,Atif Aslam,31,88,1656,1656,MOVED_DOWN,2025-02-01,113,2021-10-21
3,2026-05-03,ae,90,spotify:artist:4G9NDjRyZFDlJKMRL8hx3S,sombr,30,91,393,393,MOVED_UP,2025-08-22,196,2025-04-06
4,2026-05-03,ae,91,spotify:artist:6LEG9Ld1aLImEFEVHdWNSB,AP Dhillon,3,85,1626,1625,MOVED_DOWN,2022-03-13,196,2021-11-04
5,2026-05-03,ae,92,spotify:artist:2JEjaa7hWhE1BbL3OcoeFR,Sujatha,75,97,971,698,MOVED_UP,2026-03-11,196,2023-05-28
6,2026-05-03,ae,93,spotify:artist:2h93pZq0e7k5yf4dywlkpM,Frank Ocean,37,103,1656,1656,MOVED_UP,2023-04-17,123,2021-10-21
7,2026-05-03,ae,94,spotify:artist:6USv9qhCn6zfxlBQIYJ9qs,Dominic Fike,66,94,70,70,NO_CHANGE,2026-04-09,199,2026-02-23
8,2026-05-03,ae,95,spotify:artist:45dkTj5sMRSjrmBSBeiHym,Tate McRae,14,102,1547,962,MOVED_UP,2025-11-21,115,2021-10-21
9,2026-05-03,ae,96,spotify:artist:42k1aOQmLqy5YQmqJ2zq8s,Shaan Rahman,53,98,1456,1305,MOVED_UP,2026-03-22,198,2021-12-25


### int_charts_artists_daily

In [ ]:
# MODEL NAME: int_charts_artists_daily.sql
# Daily artist chart rows, cleaned. One row per chart_date x country x artist_uri.
# Discovery: the source sometimes lists a 2nd artist with the same name under another artist's URI.
# On those days the URI appears twice in one country, with two different chart histories
# (a chart history = artist_uri x country x entry_date). Examples: Luis Gabriel (ro),
#  Inkos'yamagcokama (za), VCT (vn).
# Decision: the oldest history owns the URI; all other histories of that URI and country
# ("side histories") are deleted. Moving them to the right URI is not reliable (see VCT).

con.sql("""
CREATE OR REPLACE VIEW int_charts_artists_daily AS
WITH histories AS (
    -- PURPOSE: turns the daily chart rows into one row per chart history. A chart history is one artist’s run on one country’s chart, and the
    -- source marks it with entry_date: every day of the same run carries the same entry_date (a re-entry can start a new one)
    -- WHY NEEDED: the model decides which rows to delete per history, not per day
    SELECT
        c.artist_uri
        , c.country_code
        , c.entry_date
        , COUNT(*)              AS chart_days -- counts how many daily rows each history has
              -- Example: VCT in Vietnam
                -- the one with URI= 7yqu… and entry date = 2021-10-21 => chart_days = 195 (this combo had 195 days in the chart)
                -- the one with URI= 7yqu… and entry date = 2026-01-21 => chart_days = 15 (this combo had 15 days in the chart)
    FROM stg_charts_artists_daily c
    GROUP BY ALL
)

, collisions AS (
    -- PURPOSE: finds the artist and country pairs where the same URI appeared more than once on at least one chart day.
    -- WHY NEEDED: It limits the side-history check to pairs that actually had a collision. Without it, side_histories would treat every
      --artist with more than one entry_date in a country as having a side history. That would be wrong for real re-entries. If an artist
      --drops off the chart and later comes back with a new entry_date, they have two histories, but never on the same day.
    SELECT DISTINCT
        c.artist_uri
        , c.country_code
    FROM stg_charts_artists_daily c
    GROUP BY -- creates one group per artist, country and day. Normally each group has one row
        c.chart_date
        , c.country_code
        , c.artist_uri
    HAVING COUNT(*) > 1 -- keeps only the groups with more than one row, which are the double days
)

, side_histories AS (
    -- PURPOSE: builds the list of histories to delete: for each pair with a collision, every history except the oldest.
    -- within a colliding artist x country, the oldest history is the real one; all others are side histories
    SELECT
        h.artist_uri
        , h.country_code
        , h.entry_date
    FROM histories h
        INNER JOIN collisions co USING (artist_uri, country_code) -- only keep the artist x country pairs that had multiple entries for any chart date
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY h.artist_uri , h.country_code
        ORDER BY
            h.entry_date            -- the oldest chart history owns the URI
            , h.chart_days DESC     -- tie-break: the longer history wins
    ) > 1 -- keep all but the oldest history (oldest is main, here we define all side histories)
)

SELECT
    c.*
FROM stg_charts_artists_daily c
    -- keep only rows that do not belong to a side history
    LEFT JOIN side_histories s
        ON  s.artist_uri   = c.artist_uri
        AND s.country_code = c.country_code
        AND s.entry_date   = c.entry_date
WHERE 0=0
    AND s.artist_uri IS NULL
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY
        c.chart_date
        , c.country_code
        , c.artist_uri
    ORDER BY c.chart_rank
) = 1 -- if dupliactes for same URI & entry date, take the row with highest rank (best position) for the given day
""")

### int_songs_weekly

In [ ]:
# MODEL NAME: int_songs_weekly.sql
# Weekly song chart per country. One row per week_start x country x song_id.
# Used by Q2 (hottest song this week), Q3 (labels with hits) and Q4 (breakouts).
# Different URIs of one song are combined here: streams are summed (they are separate real
# chart entries), rank = the best position of any of its URIs.
# Streams are only known for days in the Top 200, so a song's weekly streams = its charted streams.


con.sql("""
CREATE OR REPLACE VIEW int_songs_weekly AS
WITH params AS (
    -- Q4 breakout threshold: a song is a breakout candidate if it was outside this Top N last week
    SELECT
        20                                          AS breakout_top_n
)

, weekly AS (
    -- Weekly metrics per song x country, from the daily chart rows.
    -- int_charts_songs_daily has one row per chart_date x country x track_uri, not per song_id:
    -- two URIs of the same song on the same day are both kept (real separate chart entries).
    -- A song can therefore have several rows per day and per week, with different track_uri
    -- and sometimes slightly different names (same day, or after a URI switch during the week).
    -- Here everything is combined to one row per song_id:
    --   - track_name, artist_names: taken from the best-ranked row of the week
    --   - weekly_streams: sum over all days and all URIs of the song
    --   - best_rank: best rank (lowest number) of any URI on any day of the week
    --   - days_on_chart: distinct days, so a day with two URIs counts once
    --   - label: taken from the best-ranked row of the week that has a label
    SELECT
        d.week_start
        , d.country_code
        , d.song_id
        , ARG_MIN(d.track_name, d.chart_rank)       AS track_name        -- name from the best-ranked row
        , ARG_MIN(d.artist_names, d.chart_rank)     AS artist_names
        , ARG_MIN(d.label, d.chart_rank) FILTER (WHERE d.label IS NOT NULL) AS label  -- label from the best-ranked row that has one
        , SUM(d.streams)                            AS weekly_streams
        , MIN(d.chart_rank)                         AS best_rank
        , COUNT(DISTINCT d.chart_date)              AS days_on_chart
    FROM int_charts_songs_daily d
    GROUP BY
        d.week_start
        , d.country_code
        , d.song_id
)

-- number of chart days each country has in a week: 7 = complete week.
-- The first week (data starts on a Wednesday) and the latest week are usually incomplete,
-- and the dataset has a few missing days.
, country_weeks AS (
    SELECT
        d.week_start
        , d.country_code
        , COUNT(DISTINCT d.chart_date)              AS chart_days_in_week
    FROM int_charts_songs_daily d
    GROUP BY ALL
)

, country_weeks_prev AS (
    -- add the country's previous week, to know if it is complete too
    -- PURPOSE: adds the previous week’s values to each country’s week, so the model can check whether last week was complete.
    -- WHY NEEDED: That matters for Q4: if last week had only 4 chart days, last week’s streams look too low, and every song’s week-over-week gain looks too high.
    SELECT
        cw.*
      -- LAG(column) OVER (...) returns that column’s value from the previous row, according to the window’s order
        , LAG(cw.week_start) OVER country_order         AS prev_country_week
        , LAG(cw.chart_days_in_week) OVER country_order AS prev_chart_days_in_week
    FROM country_weeks cw
    WINDOW country_order AS (
        PARTITION BY cw.country_code -- each country is handled separately, so Estonia’s first week never looks at another country’s last week
        ORDER BY cw.week_start -- “previous row” means the previous week in time
    )
)

, with_prev AS (
    -- We have the Q4 WOW growth question, so adding the WOW growth as a metric in the model.
    -- For other period growths these logics can be defined when querying the gold layer itself.
    -- the song's previous charted week; LAG is cheaper than joining the weekly table to itself
    SELECT
        w.*
        , LAG(w.week_start) OVER song_weeks         AS prev_week_start
        , LAG(w.weekly_streams) OVER song_weeks     AS prev_weekly_streams
        , LAG(w.best_rank) OVER song_weeks          AS prev_best_rank
    FROM weekly w
    WINDOW song_weeks AS (
        PARTITION BY
            w.country_code
            , w.song_id
        ORDER BY w.week_start
    )
)

, final AS (
    SELECT
        p.week_start
        , p.country_code
        , p.song_id
        , p.track_name
        , p.artist_names
        , p.label
        , p.weekly_streams
        , p.best_rank
        , p.days_on_chart
        , cw.chart_days_in_week = 7                                                                     AS is_complete_week
        -- the country's previous week must exist, be exactly 7 days earlier and be complete
          -- prev_country_week = week_start - 7: the previous row really is last week.
          -- prev_chart_days_in_week = 7: that week has all 7 days.
        , COALESCE(cw.prev_country_week = p.week_start - 7 AND cw.prev_chart_days_in_week = 7, FALSE)   AS is_prev_week_complete
        -- previous week values only count if the song's previous chart week was exactly last week
        , CASE WHEN p.prev_week_start = p.week_start - 7 THEN p.prev_weekly_streams END                 AS prev_week_streams
        , CASE WHEN p.prev_week_start = p.week_start - 7 THEN p.prev_best_rank END                      AS prev_week_best_rank
    FROM with_prev p
        INNER JOIN country_weeks_prev cw
            ON  cw.country_code = p.country_code
            AND cw.week_start   = p.week_start
)

SELECT
    f.*
    , f.weekly_streams - COALESCE(f.prev_week_streams, 0)                                   AS stream_gain_wow   -- absolute gain, a new entry gains all its streams
    , ROUND(100.0 * (f.weekly_streams - f.prev_week_streams) / f.prev_week_streams, 1)      AS stream_growth_pct_wow
    , COALESCE(f.prev_week_best_rank, 999) > pr.breakout_top_n                              AS was_outside_top_n_prev_week  -- not charting also counts
    -- position of the song in this country & week by streams (1 = most streamed)
    , ROW_NUMBER() OVER (
        PARTITION BY
            f.country_code
            , f.week_start
        ORDER BY
            f.weekly_streams DESC
            , f.best_rank
    )                                                                                       AS streams_rank_in_week
FROM final f
  CROSS JOIN params pr        -- one row, so every row gets the same threshold
""")

### int_artist_country_weekly_presence

In [ ]:
# MODEL NAME: int_artist_country_weekly_presence.sql
# Weekly chart presence per artist x country. One row per week_start x country x artist_uri.
# presence_points = SUM(201 - rank): rank 1 = 200 points per day, rank 200 = 1 point.
# All metrics are additive (or can be rebuilt from additive ones), so any period can be
# calculated by summing weeks: e.g. Q1 over the whole window = SUM(presence_points).
# 'global' is excluded: it is a worldwide chart, not a country.

con.sql("""
CREATE OR REPLACE VIEW int_artist_country_weekly_presence AS
WITH weekly AS (
    SELECT
        CAST(DATE_TRUNC('week', c.chart_date) AS DATE)  AS week_start      -- Monday of the ISO week
        , c.country_code
        , c.artist_uri
        , ARG_MAX(c.artist_name, c.chart_date)          AS chart_artist_name   -- latest spelling on the chart
        , COUNT(*)                                      AS chart_days
        , MIN(c.chart_rank)                             AS best_rank
        , SUM(c.chart_rank)                             AS rank_sum            -- to rebuild avg_rank over any period
        , SUM(201 - c.chart_rank)                       AS presence_points -- afterwards we can do RANK() OVER (PARTITION BY artist_name, counrty ORDER BY SUM(w.presence_points) DESC)
          -- Then for whichever period we build the "presence" metric, it will be a combination of fo rhow many weeks it charted and at which rank (more weeks & better position -> stronger presence)
    FROM int_charts_artists_daily c
    WHERE 0=0
        AND c.country_code <> 'global'
    GROUP BY
        CAST(DATE_TRUNC('week', c.chart_date) AS DATE)
        , c.country_code
        , c.artist_uri
)

, country_weeks AS (
    -- number of chart days each country has in a week: 7 = complete week
    -- Calculated on a country level: the number of days with at least one artist on that country’s chart
    SELECT
        CAST(DATE_TRUNC('week', c.chart_date) AS DATE)  AS week_start
        , c.country_code
        , COUNT(DISTINCT c.chart_date)                  AS chart_days_in_week
    FROM int_charts_artists_daily c
    WHERE 0=0
        AND c.country_code <> 'global'
    GROUP BY
        CAST(DATE_TRUNC('week', c.chart_date) AS DATE)
        , c.country_code
)

SELECT
    w.week_start
    , w.country_code
    , w.artist_uri
    , COALESCE(a.artist_name, w.chart_artist_name)      AS artist_name
    , w.chart_days
    , w.best_rank
    , w.rank_sum
    , ROUND(w.rank_sum / w.chart_days, 1)               AS avg_rank
    , w.presence_points
    , cw.chart_days_in_week = 7                         AS is_complete_week
    -- 1 = the country where this artist's presence was strongest in this week
    , RANK() OVER (
        PARTITION BY w.artist_uri, w.week_start
        ORDER BY w.presence_points DESC)             AS country_rank_for_artist_in_week -- artist's presence rank per week and country
FROM weekly w
    INNER JOIN country_weeks cw
        ON  cw.country_code = w.country_code
        AND cw.week_start   = w.week_start
    LEFT JOIN stg_artists a USING (artist_uri) -- The join gives each artist one consistent name across all weeks and countries.
""")

In [ ]:
# MODEL NAME: int_song_chart_lifespan.sql
# Q5: how long a song stays on the chart after release. One row per song_id x country.
# Release date: the source release_date is the album the track is linked to NOW, which is often
# later than the real first release (discovery: Halsey "Without Me" shows 2020-01-17).
# So release = the earliest of release_date and the song's first chart entry (entry_date) anywhere.
# Only songs released on/after start_date are kept: their whole chart life is inside our data.
# is_still_charting = TRUE means the run is not finished yet (exclude these for "typical" lengths).

con.sql("""
CREATE OR REPLACE VIEW int_song_chart_lifespan AS
WITH song_release AS ( --finds each song’s release date
    SELECT
        d.song_id
        , LEAST(MIN(d.release_date), MIN(d.entry_date))    AS release_date
        -- The source release_date isn’t reliable as a first release date. It’s the date of the album the track is linked to now, which is
        -- often later than the single. For Halsey’s “Without Me” it shows 2020-01-17, although the song was on the chart from 2018.
        -- So the model takes the earlier of two dates:
            --MIN(d.release_date): the earliest release date of any of the song’s URIs.
            --MIN(d.entry_date): the first day the song appeared on any country’s chart. entry_date comes from the source and goes back before your window, so it shows when the song first charted.
      -- A song can’t chart before it’s released, so if the first chart entry is earlier, that date is the better estimate. LEAST ignores NULLs, so a
      --song without a release_date still gets its first entry_date.
    FROM int_charts_songs_daily d
    GROUP BY d.song_id
)

, runs AS (
    -- the song's chart life in each country
    SELECT
        d.song_id
        , d.country_code
        , ARG_MIN(d.track_name, d.chart_rank)               AS track_name
        , ARG_MIN(d.artist_names, d.chart_rank)             AS artist_names
        , MIN(d.chart_date)                                 AS first_chart_date
        , MAX(d.chart_date)                                 AS last_chart_date
        , COUNT(DISTINCT d.chart_date)                      AS chart_days      -- days actually on the chart (gaps excluded)
        , MIN(d.chart_rank)                                 AS best_rank
    FROM int_charts_songs_daily d
    GROUP BY ALL
)

, last_data_day AS (
    -- latest chart date per country, to spot runs that are still going
    SELECT
        d.country_code
        , MAX(d.chart_date)                                 AS last_data_date
    FROM int_charts_songs_daily d
    GROUP BY ALL
)

SELECT
    r.song_id
    , r.country_code
    , r.track_name
    , r.artist_names
    , s.release_date
    , r.first_chart_date
    , r.last_chart_date
    , r.chart_days
    , r.best_rank
    , r.first_chart_date - s.release_date                   AS days_release_to_entry   -- DATE - DATE = days
    , r.last_chart_date - s.release_date + 1                AS days_release_to_exit    -- release day counts as day 1
    , r.last_chart_date = l.last_data_date                  AS is_still_charting
FROM runs r
    INNER JOIN song_release s USING (song_id)
    INNER JOIN last_data_day l USING (country_code)
WHERE 0=0
    AND s.release_date >= DATE '2026-01-01' -- NOTE: THIS WAY OUR METRICS WORK ONLY ON NEW SONGS
""")

# int for dim tables

### int_song

In [ ]:
# Song dimension logic. One row per song_id.
# Songs come from songs.csv: one row per song, song_id = the song's main URI.
# Chart URIs missing from songs.csv (discovery: 43 URIs, no name or label) keep their own URI
# as song_id in int_charts_songs_daily. They are added here as "unknown" songs, so every
# song_id in the facts has a matching dimension row.
# Both CTEs must list the same columns in the same order (UNION ALL matches by position).


con.sql("""
CREATE OR REPLACE VIEW int_song AS
WITH known_songs AS (
    SELECT
        s.track_uri                                                     AS song_id
        , s.track_name
        , s.artist_names
        , s.artist_uris
        , s.label
        , s.release_date                                                -- source date: the album the track is linked to now
        , LEN(STRING_SPLIT(COALESCE(s.all_uris, s.track_uri), '|'))    AS uri_count
        , FALSE                                                         AS is_unknown_song
    FROM stg_songs s
)

, unknown_songs AS (
    -- song_ids on the chart that are not in songs.csv
    SELECT
        d.song_id
        , ARG_MIN(d.track_name, d.chart_rank)                           AS track_name      -- usually NULL
        , ARG_MIN(d.artist_names, d.chart_rank)                         AS artist_names    -- usually NULL
        , CAST(NULL AS VARCHAR)                                         AS artist_uris
        , ARG_MIN(d.label, d.chart_rank)                                AS label
        , MIN(d.release_date)                                           AS release_date
        , 1                                                             AS uri_count
        , TRUE                                                          AS is_unknown_song
    FROM int_charts_songs_daily d
    WHERE 0=0
        AND NOT EXISTS (
            SELECT 1
            FROM stg_songs s
            WHERE s.track_uri = d.song_id
        )
    GROUP BY d.song_id
)

SELECT * FROM known_songs
UNION ALL
SELECT * FROM unknown_songs
""")

### int_artist

In [ ]:
# Artist dimension logic. One row per artist_uri.
# Artists come from artists.csv. Chart artists missing from artists.csv are added as "unknown"
# artists with their latest chart spelling, so every artist_uri in the facts has a dimension row.
# Listener numbers from artists.csv are left out: they are a snapshot of one day and change
# daily, so they are measures, not artist attributes.
# Note: artist_name is not unique (same-name profiles exist, e.g. two Luis Gabriels).

con.sql("""
CREATE OR REPLACE VIEW int_artist AS
WITH known_artists AS (
    SELECT
        a.artist_uri
        , a.artist_name
        , FALSE                                     AS is_unknown_artist
    FROM stg_artists a
)

, unknown_artists AS (
    -- artist_uris on the chart that are not in artists.csv
    SELECT
        c.artist_uri
        , ARG_MAX(c.artist_name, c.chart_date)      AS artist_name     -- latest spelling on the chart
        , TRUE                                      AS is_unknown_artist
    FROM int_charts_artists_daily c
    WHERE 0=0
        AND NOT EXISTS (
            SELECT 1
            FROM stg_artists a
            WHERE a.artist_uri = c.artist_uri
        )
    GROUP BY c.artist_uri
)

SELECT * FROM known_artists
UNION ALL
SELECT * FROM unknown_artists
""")

# Mart (gold) layer

In [ ]:
#mart_artist_country_weekly_presence.sql
# Gold layer: Q1 - artist chart presence per country.
# All logic lives in int_artist_country_weekly_presence; this model only exposes it to reporting.

con.sql("""
CREATE OR REPLACE VIEW mart_artist_country_weekly_presence AS
SELECT
    *
FROM int_artist_country_weekly_presence
""")

In [ ]:
#mart_songs_weekly.sql
# Gold layer: Q2 hottest song this week + Q4 breakouts.
# All logic lives in int_songs_weekly; this model only exposes it to reporting.

con.sql("""
CREATE OR REPLACE VIEW mart_songs_weekly AS
SELECT
    *
FROM int_songs_weekly
""")

In [ ]:
#mart_song_chart_lifespan.sql
# Gold layer: Q5 - chart life after release per country.
# All logic lives in int_song_chart_lifespan; this model only exposes it to reporting.

con.sql("""
CREATE OR REPLACE VIEW mart_song_chart_lifespan AS
SELECT
    *
FROM int_song_chart_lifespan
""")

In [ ]:
# song dimension. All logic lives in int_song.

con.sql("""
CREATE OR REPLACE VIEW dim_song AS
SELECT
    *
FROM int_song
""")

In [ ]:
# artist dimension. All logic lives in int_artist.

con.sql("""
CREATE OR REPLACE VIEW dim_artist AS
SELECT
    *
FROM int_artist
""")

# Queries for the business questions

**Q1: For a given artist, in which country is their chart presence the strongest?**

Weekly points are summed over the whole period, then the countries are ranked. Artists are grouped by artist_uri, so two artists with the same name (like the two Luis Gabriels) stay separate.

In [ ]:
ARTIST = "Taylor Swift"

con.sql(f"""
SELECT
    a.artist_name
    , p.artist_uri
    , p.country_code
    , SUM(p.chart_days)                                         AS chart_days
    , MIN(p.best_rank)                                          AS best_rank
    , ROUND(SUM(p.rank_sum) / SUM(p.chart_days), 1)             AS avg_rank     -- day-weighted, not an average of weekly averages
    , SUM(p.presence_points)                                    AS presence_points
    , RANK() OVER (
        PARTITION BY p.artist_uri
        ORDER BY SUM(p.presence_points) DESC
    )                                                           AS country_rank_for_artist
FROM mart_artist_country_weekly_presence p
    INNER JOIN dim_artist a USING (artist_uri)
WHERE 0=0
    AND a.artist_name = '{ARTIST}'
GROUP BY
    a.artist_name
    , p.artist_uri
    , p.country_code
QUALIFY country_rank_for_artist <= 5
ORDER BY
    p.artist_uri
    , country_rank_for_artist
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,artist_name,artist_uri,country_code,chart_days,best_rank,avg_rank,presence_points,country_rank_for_artist
0,Taylor Swift,spotify:artist:06HL4z0CvFAxyc27GXpf02,ph,286.0,1,1.1,57170.0,1
1,Taylor Swift,spotify:artist:06HL4z0CvFAxyc27GXpf02,sg,286.0,1,1.7,57000.0,2
2,Taylor Swift,spotify:artist:06HL4z0CvFAxyc27GXpf02,au,286.0,1,2.0,56917.0,3
3,Taylor Swift,spotify:artist:06HL4z0CvFAxyc27GXpf02,ca,286.0,1,2.4,56808.0,4
4,Taylor Swift,spotify:artist:06HL4z0CvFAxyc27GXpf02,gb,286.0,1,2.7,56702.0,5


**Q2: What is the hottest song in Estonia this week?**

Hottest = most streams in the latest complete week (all 7 chart days).

In [ ]:
con.sql("""
SELECT
    w.week_start
    , w.streams_rank_in_week
    , s.track_name
    , s.artist_names
    , w.weekly_streams
    , w.best_rank
    , w.stream_growth_pct_wow
FROM mart_songs_weekly w
    INNER JOIN dim_song s USING (song_id)
WHERE 0=0
    AND w.country_code = 'ee'
    AND w.is_complete_week
-- keep only the latest complete week (one scan instead of a separate MAX query)
QUALIFY w.week_start = MAX(w.week_start) OVER ()
ORDER BY w.streams_rank_in_week
LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,week_start,streams_rank_in_week,track_name,artist_names,weekly_streams,best_rank,stream_growth_pct_wow
0,2026-10-19,1,Housewife,Tate McRae,63620.0,1,359.3
1,2026-10-19,2,n ä e m e . v e e l,SADU,53892.0,1,110.0
2,2026-10-19,3,rongikupee,säm,31052.0,3,-4.5
3,2026-10-19,4,p õ u d,SADU,24413.0,4,28.4
4,2026-10-19,5,lainte laul,nublu|TIEF CITY 44,23783.0,4,-7.9


**Q3: Which record labels are consistently producing hits?**

A hit is a song’s first week in the Top 10 of any chart. Each hit counts once, in the month of that week’s Monday. Consistency is the share of months with at least one new hit.

In [ ]:
con.sql("""
WITH first_hit AS (
    -- each hit song once: the first week it reached the Top 10 in any chart
    SELECT
        w.song_id
        , LOWER(w.label)                                        AS label_key
        , w.label
        , CAST(DATE_TRUNC('month', w.week_start) AS DATE)       AS hit_month   -- month of the week's Monday
    FROM mart_songs_weekly w
    WHERE 0=0
        AND w.best_rank <= 10
        AND w.label IS NOT NULL
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY w.song_id
        ORDER BY
            w.week_start
            , w.best_rank
    ) = 1
)

, period AS (
    -- number of months in the data
    SELECT
        COUNT(DISTINCT DATE_TRUNC('month', w.week_start))       AS months_in_period
    FROM mart_songs_weekly w
)

SELECT
    MODE(f.label)                                               AS label_name   -- most common spelling
    , COUNT(*)                                                  AS hit_songs
    , COUNT(DISTINCT f.hit_month)                               AS months_with_new_hit
    , p.months_in_period
    , ROUND(100.0 * COUNT(DISTINCT f.hit_month) / p.months_in_period, 1)
                                                                AS pct_months_with_new_hit
FROM first_hit f
    CROSS JOIN period p
GROUP BY
    f.label_key
    , p.months_in_period
ORDER BY
    months_with_new_hit DESC
    , hit_songs DESC
LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,label_name,hit_songs,months_with_new_hit,months_in_period,pct_months_with_new_hit
0,WM Finland,112,11,11,100.0
1,Alda Music,106,11,11,100.0
2,WEA Latina,61,11,11,100.0
3,Universal Music Oy,35,11,11,100.0
4,Astro Music Group,33,11,11,100.0
5,Minos - EMI SA Distribution Deals,71,10,11,90.9
6,Warner Music,51,10,11,90.9
7,Columbia,48,10,11,90.9
8,Sony Music Entertainment,43,10,11,90.9
9,Atlantic Records,26,10,11,90.9


**Q4: Which songs are breaking out?**

The largest week-over-week stream gain among songs outside the Top 20 last week, or not on the chart. It uses the latest week where both this week and last week are complete.

In [ ]:
con.sql("""
SELECT
    w.week_start
    , w.country_code
    , s.track_name
    , s.artist_names
    , w.prev_week_best_rank
    , w.best_rank
    , w.prev_week_streams
    , w.weekly_streams
    , w.stream_gain_wow
FROM mart_songs_weekly w
    INNER JOIN dim_song s USING (song_id)
WHERE 0=0
    AND w.country_code <> 'global'
    AND w.is_complete_week
    AND w.is_prev_week_complete
    AND w.was_outside_top_n_prev_week
QUALIFY w.week_start = MAX(w.week_start) OVER ()
ORDER BY w.stream_gain_wow DESC
LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,week_start,country_code,track_name,artist_names,prev_week_best_rank,best_rank,prev_week_streams,weekly_streams,stream_gain_wow
0,2026-10-19,br,Flash (Minha Mente) 💠,Hugo & Guilherme|Zé Vaqueiro,21,3,1315215.0,5771050.0,4455835.0
1,2026-10-19,us,Sports car,Tate McRae,<NA>,56,NaN,2723344.0,2723344.0
2,2026-10-19,br,Ritmo Era Uma Vez,MC VN Cria|MC BF|DJ Oreia 074|Fortuna Music Group,<NA>,41,NaN,2614976.0,2614976.0
3,2026-10-19,mx,Cansada,Victor Mendivil|Victoria Mendivil,<NA>,4,NaN,2605101.0,2605101.0
4,2026-10-19,br,Solinho do Fora PT,Prismah,141,3,186452.0,2761332.0,2574880.0
5,2026-10-19,br,"Bandoleira (feat. MC Meno K, Mc Lele JP & Mc J...",MC Ryan SP|Nego Trufa|DJ Glenner|Dj Daltin no ...,60,23,1063571.0,3338887.0,2275316.0
6,2026-10-19,us,Coming Up Roses,Harry Styles,188,91,277735.0,2390035.0,2112300.0
7,2026-10-19,mx,FES,Victor Mendivil|Omar Camacho,<NA>,6,NaN,2016720.0,2016720.0
8,2026-10-19,br,FUNK NIKOLAS FERREIRA,Nikolas Ferreira,<NA>,6,NaN,1948945.0,1948945.0
9,2026-10-19,id,"Hidup Untukmu, Mati Tanpamu",Noah,167,63,212213.0,2025042.0,1812829.0


**Q5: How long does a song typically stay on the chart after release, and does it differ by country?**

Songs released in the window whose chart run has ended. Medians, because a few long-running hits pull the average up.

In [ ]:
con.sql("""
SELECT
    l.country_code
    , COUNT(*)                                  AS songs
    , MEDIAN(l.chart_days)                      AS median_chart_days
    , MEDIAN(l.days_release_to_exit)            AS median_days_release_to_exit
    , MEDIAN(l.days_release_to_entry)           AS median_days_release_to_entry
    , ROUND(AVG(l.chart_days), 1)               AS avg_chart_days
FROM mart_song_chart_lifespan l
WHERE 0=0
    AND NOT l.is_still_charting
GROUP BY l.country_code
ORDER BY median_chart_days DESC
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,country_code,songs,median_chart_days,median_days_release_to_exit,median_days_release_to_entry,avg_chart_days
0,eg,184,15.5,26.5,1.0,34.1
1,vn,257,14.0,22.0,1.0,25.3
2,in,161,12.0,27.0,6.0,26.5
3,jp,204,12.0,23.0,0.0,28.4
4,th,176,11.0,20.0,1.0,23.8
...,...,...,...,...,...,...
68,is,604,2.0,7.0,0.0,12.2
69,il,41,2.0,4.0,1.0,4.1
70,by,35,1.0,2.0,0.0,2.7
71,ch,746,1.0,3.0,0.0,8.9
